In [6]:
# Đường dẫn chính xác theo ảnh cấu trúc Google Drive của bạn
file_path = '/content/drive/MyDrive/Nhap_Mon_Hoc_May/Buoi2/Du_lieu_thuc_hanh_chuan_bi_du_lieu.xlsx'

# Đọc sheet Du_lieu_tho
df_raw = pd.read_excel(file_path, sheet_name='Du_lieu_tho')
df = df_raw.copy()

In [8]:
# ==========================================
# BƯỚC 1: KẾT NỐI DRIVE & ĐỌC DỮ LIỆU THÔ
# ==========================================
import pandas as pd
import numpy as np
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# Đường dẫn chính xác tới file trong thư mục Buoi2
file_path = '/content/drive/MyDrive/Nhap_Mon_Hoc_May/Buoi2/Du_lieu_thuc_hanh_chuan_bi_du_lieu.xlsx'

# Đọc sheet 'Du_lieu_tho'
df_raw = pd.read_excel(file_path, sheet_name='Du_lieu_tho')

# Tạo bản sao DataFrame để xử lý (không đè dữ liệu gốc)
df = df_raw.copy()

print("=== 1. TẢI DỮ LIỆU THÀNH CÔNG ===")
print(f"Kích thước ban đầu: {df.shape[0]} dòng, {df.shape[1]} cột\n")


# ==========================================
# BÀI 2: LẬP BÁO CÁO DỮ LIỆU THIẾU VÀ TRÙNG LẶP
# ==========================================
print("--- 2. BÁO CÁO DỮ LIỆU THIẾU CỦA TỪNG CỘT ---")
missing_report = pd.DataFrame({
    'Số lượng thiếu': df.isnull().sum(),
    'Tỷ lệ thiếu (%)': (df.isnull().mean() * 100).round(2)
}).sort_values(by='Số lượng thiếu', ascending=False)
print(missing_report)

print("\n--- ĐẾM TRÙNG LẶP ---")
print(f"Số dòng trùng hoàn toàn: {df.duplicated().sum()}")
print(f"Số dòng trùng Product_ID: {df.duplicated(subset=['Product_ID'], keep=False).sum()}\n")


# ==========================================
# BƯỚC 3: XỬ LÝ LÀM SẠCH DỮ LIỆU (PIPELINE)
# ==========================================

# 1. Xóa dòng trùng lặp hoàn toàn
df.drop_duplicates(inplace=True)

# 2. Xử lý trùng khóa Product_ID (Giữ lại dòng xuất hiện đầu tiên)
df.drop_duplicates(subset=['Product_ID'], keep='first', inplace=True)

# 3. Chuẩn hóa chuỗi (chuyển các cột dạng chữ về chữ thường và xóa khoảng trắng thừa)
text_columns = df.select_dtypes(include=['object']).columns
for col in text_columns:
    df[col] = df[col].astype(str).str.strip().str.lower()

# 4. Xử lý dữ liệu thiếu (Imputation)
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True) # Điền giá trị trung vị cho cột số

categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True) # Điền yếu vị (mode) cho cột phân loại

print("=== 3. ĐÃ XỬ LÝ LÀM SẠCH VÀ ĐIỀN KHUYẾT DỮ LIỆU ===")
print(f"Kích thước sau khi làm sạch: {df.shape[0]} dòng, {df.shape[1]} cột\n")


# ==========================================
# BƯỚC 4: LƯU FILE KẾT QUẢ ĐÃ LÀM SẠCH VÀO DRIVE
# ==========================================
output_path = '/content/drive/MyDrive/Nhap_Mon_Hoc_May/Buoi2/Du_lieu_da_lam_sach.xlsx'

# Lưu kết quả ra file Excel mới trong thư mục Buoi2
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Du_lieu_sach', index=False)
    missing_report.to_excel(writer, sheet_name='Bao_cao_du_lieu_thieu')

print(" ĐÃ LƯU THÀNH CÔNG FILE KẾT QUẢ VÀO DRIVE:")
print(f" Đường dẫn: {output_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== 1. TẢI DỮ LIỆU THÀNH CÔNG ===
Kích thước ban đầu: 228 dòng, 14 cột

--- 2. BÁO CÁO DỮ LIỆU THIẾU CỦA TỪNG CỘT ---
                   Số lượng thiếu  Tỷ lệ thiếu (%)
Description                    15             6.58
Rating                         13             5.70
Price_VND                       9             3.95
Seller_City                     7             3.07
Created_Date                    6             2.63
Product_Name                    0             0.00
Purchase_Count                  0             0.00
Category                        0             0.00
Product_ID                      0             0.00
Stock                           0             0.00
Weight                          0             0.00
Has_Image                       0             0.00
Discount                        0             0.00
Target_High_Sales               0      